In [ ]:
%pip install azure-ai-evaluation[redteam] azure-identity httpx dotenv 

In [ ]:
import os
import argparse
from typing import Optional, Dict
import argparse
from azure.identity import DefaultAzureCredential
from azure.ai.evaluation.red_team import RedTeam, RiskCategory, AttackStrategy, SupportedLanguages  # [1](https://github.com/MicrosoftDocs/azure-ai-docs/blob/main/articles/foundry-classic/how-to/develop/run-scans-ai-red-teaming-agent.md)
from os.path import join
from dotenv import load_dotenv
from azure.ai.ml import MLClient
from azure.ai.ml.entities import Model
from azure.identity import DefaultAzureCredential


In [ ]:

def build_azure_openai_config() -> Dict[str, str]:
    # Supported target: model configuration dict (Azure OpenAI) [1](https://github.com/MicrosoftDocs/azure-ai-docs/blob/main/articles/foundry-classic/how-to/develop/run-scans-ai-red-teaming-agent.md)
    return {
#        "azure_endpoint": os.environ["AZURE_OPENAI_ENDPOINT"],
#        "api_key": os.environ["AZURE_OPENAI_KEY"],
#        "azure_deployment": os.environ["AZURE_OPENAI_DEPLOYMENT"],
        "azure_endpoint": "https://foundrydevpub6741.cognitiveservices.azure.com/",
        "api_key": "",
        "azure_deployment": "gpt-4.1-mini",
        "api_version": "2024-12-01-preview"

    }

def make_simple_rest_callback(endpoint_url: str, bearer_token: Optional[str] = None):
    """
    Suggestion (not from docs): implement calling your Azure ML online endpoint (or any REST endpoint).
    The RedTeam SDK supports a simple callback signature: (query: str) -> str [1](https://github.com/MicrosoftDocs/azure-ai-docs/blob/main/articles/foundry-classic/how-to/develop/run-scans-ai-red-teaming-agent.md)
    """
    import httpx

    headers = {"Content-Type": "application/json"}
    if bearer_token:
        print("Using bearer token for authentication in REST callback")
        headers["Authorization"] = f"Bearer {bearer_token}"

    def callback(query: str) -> str:
        payload = {
                    "text": query
                }

        r = httpx.post(endpoint_url, json=payload, headers=headers, timeout=60)
        r.raise_for_status()
        data = r.json()
        # adjust to your endpoint schema:

        return data["generated_text"]  if isinstance(data, dict) else str(data)
        # return data.get("output", "") if isinstance(data, dict) else str(data)

    return callback

# --- Runner ------------------------------------------------------------------


def parse_args(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--mode", choices=["azure_openai_config", "rest_callback"], required=True)
    p.add_argument("--output_path", default="redteam_scorecard.json")
    p.add_argument("--scan_name", default="RedTeam Scan")
    p.add_argument("--num_objectives", type=int, default=10)
    p.add_argument("--language", default="English")
    p.add_argument("--risk_categories", default="Violence,HateUnfairness,Sexual,SelfHarm")
    p.add_argument("--attack_strategies", default="EASY,MODERATE,DIFFICULT")
    p.add_argument("--rest_endpoint_url", default=None)
    return p.parse_args(argv)


def to_risk_categories(csv: str):
    mapping = {
        "Violence": RiskCategory.Violence,
        "HateUnfairness": RiskCategory.HateUnfairness,
        "Sexual": RiskCategory.Sexual,
        "SelfHarm": RiskCategory.SelfHarm,
        "ProtectedMaterial": RiskCategory.ProtectedMaterial,
        "CodeVulnerability": RiskCategory.CodeVulnerability,
        "UngroundedAttributes": RiskCategory.UngroundedAttributes,
    }
    return [mapping[x.strip()] for x in csv.split(",") if x.strip()]

def to_attack_strategies(csv: str):
    mapping = {
        "EASY": AttackStrategy.EASY,
        "MODERATE": AttackStrategy.MODERATE,
        "DIFFICULT": AttackStrategy.DIFFICULT,
    }
    return [mapping[x.strip()] for x in csv.split(",") if x.strip()]

def to_language(name: str):
    # SupportedLanguages is documented; list includes French, etc. [1](https://github.com/MicrosoftDocs/azure-ai-docs/blob/main/articles/foundry-classic/how-to/develop/run-scans-ai-red-teaming-agent.md)
    return getattr(SupportedLanguages, name, SupportedLanguages.English)

In [ ]:
def ensure_https(url: str, name: str) -> str:
    print(f"url: {url}")
    if url.startswith("http://"):
        print(f"WARNING: {name} uses http:// — upgrading to https:// (bearer token auth requires TLS)")
        return "https://" + url[len("http://"):]
    if not url.startswith("https://"):
        raise ValueError(f"{name} must be an https:// URL, got: {url!r}")
    return url

async def main(argv=None):
    args = argv if isinstance(argv, argparse.Namespace) else parse_args(argv)
    dotenv_path = join(os.path.abspath(''), '.env')
    print(dotenv_path)
    load_dotenv(dotenv_path)

    # Foundry project identification supported as hub-style dict or AZURE_AI_PROJECT URL [1](https://github.com/MicrosoftDocs/azure-ai-docs/blob/main/articles/foundry-classic/how-to/develop/run-scans-ai-red-teaming-agent.md)
    # azure_ai_project: Any = os.environ.get("AZURE_AI_PROJECT")
    # if azure_ai_project:
    print("Using AZURE_AI_PROJECT from environment")
    # azure_ai_project: Any = os.environ.get("AZURE_AI_PROJECT")
    # azure_ai_project = "https://foundrydevpub6741.services.ai.azure.com/api/projects/foundrydevpub6741-project"
    azure_ai_project = "https://cog-standard-tgug2.services.ai.azure.com/api/projects/default-project"
    # azure_ai_project = ensure_https(azure_ai_project, "AZURE_AI_PROJECT")
    # else:
    # print("Using AZURE_SUBSCRIPTION_ID, AZURE_RESOURCE_GROUP, and AZURE_PROJECT_NAME from environment")
    # azure_ai_project = {
    #     "subscription_id": os.environ["AZURE_SUBSCRIPTION_ID"],
    #     "resource_group_name": os.environ["AZURE_RESOURCE_GROUP"],
    #     "project_name": os.environ["AZURE_PROJECT_NAME"],
    # }
    credential = DefaultAzureCredential()

    red_team_agent = RedTeam(
        azure_ai_project=azure_ai_project,
        credential=credential,
        risk_categories=to_risk_categories(args.risk_categories),
        num_objectives=args.num_objectives,
        language=to_language(args.language),
    )  # [1](https://github.com/MicrosoftDocs/azure-ai-docs/blob/main/articles/foundry-classic/how-to/develop/run-scans-ai-red-teaming-agent.md)

    attack_strategies = to_attack_strategies(args.attack_strategies)

    if args.mode == "azure_openai_config":
        target = build_azure_openai_config()
        # target["azure_endpoint"] = ensure_https(target["azure_endpoint"], "AZURE_OPENAI_ENDPOINT")
    else:
        if not args.rest_endpoint_url:
            raise ValueError("--rest_endpoint_url is required for rest_callback mode")
        # ensure_https(args.rest_endpoint_url, "rest_endpoint_url")
        # Fetch endpoint credentials


        subscription_id = "4e80bb93-03b8-4aaa-8394-7c52710f434a"
        resource_group = "ces3-posttraining"
        workspace_name = "testwest"

        ml_client = MLClient(
            DefaultAzureCredential(),
            subscription_id,
            resource_group,
            workspace_name,
        )


        creds = ml_client.online_endpoints.get_keys(
                        name="hf-opt-endpoint-20260324215047"
                )

        # API keys (for auth_mode: key)
        primary_key = creds.primary_key
        secondary_key = creds.secondary_key
        access_token_str = primary_key
        # token = credential.get_token("https://cognitiveservices.azure.com/.default")
        # access_token_str = token.token
        # access_token_str =""
        target = make_simple_rest_callback(args.rest_endpoint_url,access_token_str)

    result = await red_team_agent.scan(
        target=target,
        scan_name=args.scan_name,
        attack_strategies=attack_strategies,
        output_path=args.output_path,
    )  # [1](https://github.com/MicrosoftDocs/azure-ai-docs/blob/main/articles/foundry-classic/how-to/develop/run-scans-ai-red-teaming-agent.md)

    # The scan writes the scorecard JSON to output_path; result also includes row-level data. [1](https://github.com/MicrosoftDocs/azure-ai-docs/blob/main/articles/foundry-classic/how-to/develop/run-scans-ai-red-teaming-agent.md)
    print(f"Saved scorecard to: {args.output_path}")
    return result

args = parse_args([
    #"--mode", "azure_openai_config",
    "--mode","rest_callback",
    "--output_path", "outputs/redteam_scorecard_opt_350m_2",
    "--scan_name", "CI RedTeam Scan",
#    "--num_objectives", "10",
    "--num_objectives", "2",
    "--risk_categories", "Violence,HateUnfairness,Sexual,SelfHarm",
#    "--risk_categories", "Violence,HateUnfairness,Sexual",
#    "--risk_categories", "Violence",
    "--attack_strategies", "EASY,MODERATE",
#    "--attack_strategies", "EASY",
   # "--rest_endpoint_url","https://foundrydevpub6741.cognitiveservices.azure.com/openai/deployments/gpt-4.1-mini/chat/completions?api-version=2025-01-01-preview"
    #"--rest_endpoint_url","https://foundrydevpub6741.cognitiveservices.azure.com/openai/deployments/gpt-4.1-mini/chat/completions?api-version=2025-01-01-preview"
    #"--rest_endpoint_url","https://cog-standard-tgug2.cognitiveservices.azure.com/openai/deployments/gpt-4.1-mini/chat/completions?api-version=2025-01-01-preview"
    "--rest_endpoint_url","https://hf-opt-endpoint-20260324215047.westus.inference.ml.azure.com/score"
])

res = await main(args)
